[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-function_calling/examples/function_calling/function_calling.ipynb)

# Call an ordinary function

The assistant can check the weather, change a tariff, look up an order or read a balance. The arguments that come from a closed set are questions. The arguments that are numbers are a regular expression. An LLM would emit a JSON tool call you then validate. FRIDA-Decisions picks the function and each closed argument in one request; code calls a normal Python function, or asks a follow-up when `confidence` is low.

Every argument question is asked even when the function does not need it. Unused answers are ignored. That is speculative fan-out: extra questions are cheap next to one copy of the text, and the code decides which of them matter.

Pattern: [Function calling](https://docs.typesafe.ai/cookbooks/function_calling).


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
import re

FUNCTIONS = {
    "weather": "прогноз погоды в названном городе",
    "tariff": "сменить тариф мобильной связи",
    "order": "узнать статус заказа по его номеру",
    "balance": "узнать остаток денег на счёте или карте",
}
CITY = {
    "msk": "Москва",
    "spb": "Санкт-Петербург",
    "kzn": "Казань",
    "sochi": "Сочи",
    "none": "город не назван",
}
PLAN = {
    "basic": "базовый тариф, мало интернета",
    "smart": "средний тариф с пакетом минут и гигабайт",
    "unlimited": "безлимитный интернет",
    "none": "тариф не назван",
}
PERIOD = {
    "month": "на один месяц",
    "year": "сразу на год",
    "none": "срок не назван",
}

MIN_CONFIDENCE = 0.55
ORDER_RE = re.compile(r"(?:заказ\w*|№)\s*№?\s*(\d{3,})", re.IGNORECASE)


def request_for(text: str) -> dict:
    return {
        "state": text,
        "questions": {
            "fn": {"type": "choice", "instructions": "Какую функцию вызвать?", "criteria": FUNCTIONS},
            "city": {"type": "choice", "instructions": "О каком городе речь? Если город не назван, выбери none.",
                     "criteria": CITY},
            "plan": {"type": "choice", "instructions": "Какой тариф назван? Если не назван, выбери none.",
                     "criteria": PLAN},
            "period": {"type": "choice", "instructions": "На какой срок менять тариф? Если срок не назван, выбери none.",
                       "criteria": PERIOD},
        },
    }


def weather(city: str) -> str:
    return f"Завтра в городе {CITY[city]} без осадков."


def change_tariff(plan: str, period: str) -> str:
    return f"Заявка: тариф {PLAN[plan]}, срок {PERIOD[period]}."


def order_status(number: str) -> str:
    return f"Заказ {number}: передан в доставку."


def balance() -> str:
    return "Остаток на счёте: 1 240 рублей."


def act(text: str, answers: dict) -> dict:
    fn = answers["fn"]
    if fn["confidence"] < MIN_CONFIDENCE:
        return {"outcome": "clarify", "ask": "Уточните: погода, тариф, статус заказа или баланс.",
                "confidence": fn["confidence"]}
    name = fn["choice"]
    if name == "weather":
        city = answers["city"]["choice"]
        if city == "none":
            return {"outcome": "clarify", "ask": "В каком городе посмотреть погоду?",
                    "confidence": fn["confidence"]}
        return {"outcome": f"weather:{city}", "result": weather(city), "confidence": fn["confidence"]}
    if name == "tariff":
        plan, period = answers["plan"]["choice"], answers["period"]["choice"]
        if plan == "none" or period == "none":
            return {"outcome": "clarify", "ask": "Какой тариф и на какой срок?",
                    "confidence": fn["confidence"]}
        return {"outcome": f"tariff:{plan}:{period}", "result": change_tariff(plan, period),
                "confidence": fn["confidence"]}
    if name == "order":
        found = ORDER_RE.search(text)
        if not found:
            return {"outcome": "clarify", "ask": "Назовите номер заказа.", "confidence": fn["confidence"]}
        return {"outcome": f"order:{found.group(1)}", "result": order_status(found.group(1)),
                "confidence": fn["confidence"]}
    return {"outcome": "balance", "result": balance(), "confidence": fn["confidence"]}


## Code calls the function

`outcome` is what the code did: a concrete call, or `clarify`. The gold column is the same shape, so the accuracy is the share of utterances where the code did the labelled thing.


In [4]:
def run(judge, rows: list[dict]) -> None:
    hits = 0
    for row in rows:
        response = judge(request_for(row["text"]))
        decision = act(row["text"], response["answers"])
        hits += decision["outcome"] == row["outcome"]
        detail = decision.get("result") or decision.get("ask")
        print(f"{'ok' if decision['outcome'] == row['outcome'] else 'miss':4} "
              f"got {decision['outcome']:22} gold {row['outcome']:22} "
              f"p={decision['confidence']:.2f}  {detail}")
    print(f"\n{hits}/{len(rows)} outcomes match the labels in data.json")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-function-calling/examples/function_calling/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


ok   got weather:kzn            gold weather:kzn            p=1.00  Завтра в городе Казань без осадков.
ok   got tariff:unlimited:year  gold tariff:unlimited:year  p=1.00  Заявка: тариф безлимитный интернет, срок сразу на год.
ok   got order:5512             gold order:5512             p=1.00  Заказ 5512: передан в доставку.
ok   got balance                gold balance                p=1.00  Остаток на счёте: 1 240 рублей.
ok   got clarify                gold clarify                p=1.00  Какой тариф и на какой срок?
ok   got clarify                gold clarify                p=1.00  В каком городе посмотреть погоду?
ok   got order:8841             gold order:8841             p=1.00  Заказ 8841: передан в доставку.
ok   got balance                gold balance                p=1.00  Остаток на счёте: 1 240 рублей.

8/8 outcomes match the labels in data.json
